In [1]:
pip install jira

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 4.2 MB/s eta 0:00:00


In [14]:
#!/usr/bin/env python3
"""
JSM Intelligent Support Automation Platform (Final - C and C)

Modes selected:
- Log source pattern: C (try both)
  A) /<processserver>/<tenant>/<store>/<device>/
  B) /<processserver>/<sub_path>/<uuid_or_device>/
- OCR mode: C (use easyocr if installed, else fallback to ticket text only)

Input:
- Ticket number only (Jira issue key)

Flow:
1) Fetch Jira ticket (summary, description, comments, attachments)
2) OCR screenshots (optional if easyocr available)
3) Parse env/tenant/store/processserver/device/uuid/date from full context
4) Try log routing patterns A + B dynamically
5) Ticket classification + emotion + response
6) Ticket-relevant log analysis + RCA
7) KB generation
8) Incident prediction
9) Jira ticket update
"""

import os
import re
import sys
import io
import gzip
import time
import json
import logging
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional, Tuple
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata

import numpy as np
import pandas as pd
from dotenv import load_dotenv
import os
from google.colab import userdata

os.environ["JIRA_URL"] = "https://checkpt.atlassian.net"
os.environ["JIRA_EMAIL"] = userdata.get("JIRA_EMAIL")          # add in Colab Secrets
os.environ["JIRA_API_TOKEN"] = userdata.get("JIRA_API_TOKEN")  # add in Colab Secrets

# Optional dependencies
try:
    import requests
    HAS_REQUESTS = True
except Exception:
    HAS_REQUESTS = False

try:
    from bs4 import BeautifulSoup
    HAS_BS4 = True
except Exception:
    HAS_BS4 = False

try:
    import torch
    from transformers import pipeline
    HAS_TRANSFORMERS = True
    NLP_DEVICE = 0 if torch.cuda.is_available() else -1
except Exception:
    HAS_TRANSFORMERS = False
    NLP_DEVICE = -1

try:
    from jira import JIRA
    HAS_JIRA = True
except Exception:
    HAS_JIRA = False

try:
    import easyocr
    HAS_EASYOCR = True
except Exception:
    HAS_EASYOCR = False

# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("jsm-final-cc")

load_dotenv()

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
class Config:
    def __init__(self):
        # Jira
        self.JIRA_URL = os.getenv("JIRA_URL", "https://checkpt.atlassian.net")
        self.JIRA_EMAIL = os.getenv("JIRA_EMAIL")
        self.JIRA_API_TOKEN = os.getenv("JIRA_API_TOKEN")
        self.JIRA_PROJECT_KEY = os.getenv("JIRA_PROJECT_KEY", "ItemOptix Checkpoint Support")

        # Environment roots
        self.ENV_URLS = {
            "prod": os.getenv("LOG_ROOT_PROD", "https://files-io-prod.eus1-n.itemoptix.com/"),
            "ovs": os.getenv("LOG_ROOT_OVS", "https://files-io-ovsprod.eus1-n.itemoptix.com/"),
            "preprod": os.getenv("LOG_ROOT_PREPROD", "https://files-io-preprod.eus1-n.itemoptix.com/"),
            "pr": os.getenv("LOG_ROOT_PR", "https://files-io-pr.eus1-n.itemoptix.com/"),
            "europe": os.getenv("LOG_ROOT_EUROPE", "https://files-io-prod.eu1-n.itemoptix.com/"),
            "prodmr": os.getenv("LOG_ROOT_PRODMR", "https://files-io-prodmr.eu1-n.itemoptix.com/"),
        }
        self.DEFAULT_ENV = os.getenv("DEFAULT_ENV", "preprod")
        self.DEFAULT_ROOT = self.ENV_URLS.get(self.DEFAULT_ENV, self.ENV_URLS["preprod"])

        self.PROCESS_SERVERS = [s.strip() for s in os.getenv(
            "PROCESS_SERVERS",
            "processserver-0,processserver-1,processserver-2,processserver-3,processserver-4"
        ).split(",") if s.strip()]

        # Optional known subpaths to try for pattern B
        self.DEFAULT_SUB_PATHS = [p.strip() for p in os.getenv(
            "DEFAULT_SUB_PATHS",
            "/jpretailerspp/5.11_TACTICAL_STORE_TOKYO/"
        ).split(",") if p.strip()]

        self.MIN_VALID_SIZE = int(os.getenv("MIN_VALID_SIZE", "1"))
        self.ATTACHMENT_DIR = os.getenv("ATTACHMENT_DIR", "ticket_attachments")
        self.DOWNLOAD_DIR = os.getenv("DOWNLOAD_DIR", "downloaded_logs")
        self.DEFAULT_LOOKBACK_DAYS = int(os.getenv("DEFAULT_LOOKBACK_DAYS", "2"))

config = Config()

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def dedup_keep_order(items: List[str]) -> List[str]:
    seen, out = set(), []
    for x in items:
        v = (x or "").strip()
        if v and v not in seen:
            seen.add(v)
            out.append(v)
    return out

def normalize_hyphen_spaces(text: str) -> str:
    return re.sub(r"\s*-\s*", "-", text or "")

def default_date_candidates(n_days: int = 2) -> List[str]:
    now = datetime.now()
    return [(now - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(max(1, n_days))]

def no_cache_get(url: str, timeout: int = 20):
    if not HAS_REQUESTS:
        raise RuntimeError("requests not installed")
    headers = {
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "If-None-Match": "",
        "If-Modified-Since": "",
    }
    return requests.get(url, headers=headers, timeout=timeout)

# ---------------------------------------------------------------------------
# Jira
# ---------------------------------------------------------------------------
class JiraIntegration:
    def __init__(self, url: str, email: Optional[str], token: Optional[str], project: str):
        self.url = url
        self.project = project
        self.jira = None
        if not HAS_JIRA:
            logger.warning("jira package not installed.")
            return
        if not email or not token:
            logger.warning("Jira credentials missing.")
            return
        try:
            self.jira = JIRA(server=url, basic_auth=(email, token))
            logger.info("Connected to Jira.")
        except Exception as e:
            logger.warning(f"Jira connection failed: {e}")

    def is_ready(self) -> bool:
        return self.jira is not None

    def fetch_ticket_context(self, issue_key: str) -> Dict[str, Any]:
        if not self.jira:
            raise RuntimeError("Jira not connected.")
        issue = self.jira.issue(issue_key)
        summary = getattr(issue.fields, "summary", "") or ""
        description = getattr(issue.fields, "description", "") or ""
        reporter = getattr(getattr(issue.fields, "reporter", None), "displayName", "Customer")
        status = getattr(getattr(issue.fields, "status", None), "name", "Unknown")
        priority = getattr(getattr(issue.fields, "priority", None), "name", "N/A")

        comments = []
        try:
            for c in self.jira.comments(issue):
                comments.append(getattr(c, "body", "") or "")
        except Exception:
            pass

        attachments = []
        for a in getattr(issue.fields, "attachment", []):
            attachments.append({
                "filename": getattr(a, "filename", ""),
                "content": getattr(a, "content", ""),
                "mimeType": getattr(a, "mimeType", "")
            })

        return {
            "issue": issue,
            "key": issue_key,
            "summary": summary,
            "description": description,
            "reporter": reporter,
            "status": status,
            "jira_priority": priority,
            "comments": comments,
            "attachments": attachments
        }

    def update_ticket(self, issue_key: str, priority: str, labels: List[str], comment: str) -> bool:
        if not self.jira:
            return False
        try:
            issue = self.jira.issue(issue_key)
            pmap = {"P1": "1", "P2": "2", "P3": "3", "P4": "4", "P5": "5"}
            issue.update(fields={"priority": {"id": pmap.get(priority, "3")}, "labels": dedup_keep_order(labels)})
            self.jira.add_comment(issue_key, comment)
            return True
        except Exception as e:
            logger.warning(f"Failed Jira update {issue_key}: {e}")
            return False

# ---------------------------------------------------------------------------
# OCR (C: use if available else fallback)
# ---------------------------------------------------------------------------
class ScreenshotOCR:
    def __init__(self, attachment_dir: str):
        self.attachment_dir = attachment_dir
        os.makedirs(self.attachment_dir, exist_ok=True)
        self.reader = easyocr.Reader(["en"]) if HAS_EASYOCR else None
        if not HAS_EASYOCR:
            logger.info("easyocr unavailable, OCR fallback to text-only mode.")

    @staticmethod
    def is_image(filename: str) -> bool:
        f = (filename or "").lower()
        return f.endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp"))

    def download_images(self, jira_client, issue) -> List[str]:
        paths = []
        if not HAS_REQUESTS:
            return paths
        for att in getattr(issue.fields, "attachment", []):
            fn = getattr(att, "filename", "")
            if not self.is_image(fn):
                continue
            url = getattr(att, "content", "")
            if not url:
                continue
            local = os.path.join(self.attachment_dir, f"{issue.key}_{int(time.time()*1000)}_{fn}")
            try:
                r = jira_client._session.get(url, stream=True, timeout=30)
                r.raise_for_status()
                with open(local, "wb") as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
                paths.append(local)
            except Exception as e:
                logger.warning(f"Attachment download failed: {fn} -> {e}")
        return paths

    def ocr_images(self, image_paths: List[str]) -> str:
        if not self.reader or not image_paths:
            return ""
        texts = []
        for p in image_paths:
            try:
                lines = self.reader.readtext(p, detail=0, paragraph=True)
                txt = "\n".join(lines).strip()
                if txt:
                    texts.append(f"[OCR:{os.path.basename(p)}]\n{txt}")
            except Exception as e:
                logger.warning(f"OCR failed for {p}: {e}")
        return "\n\n".join(texts)

    def cleanup(self, image_paths: List[str]):
        for p in image_paths:
            try:
                if os.path.exists(p):
                    os.remove(p)
            except Exception:
                pass

# ---------------------------------------------------------------------------
# Routing/entity extraction
# ---------------------------------------------------------------------------
def parse_routing_entities(text: str, cfg: Config) -> Dict[str, Any]:
    t = normalize_hyphen_spaces(text or "")
    tl = t.lower()

    # env URL
    env_urls = dedup_keep_order(re.findall(r"https?://files-io-[a-z0-9-]+\.[a-z0-9\.-]+/", tl))

    # env names
    env_names = []
    env_names += re.findall(r"\b(?:env|environment)\s*[:=]\s*([a-z0-9_-]+)\b", tl)
    for e in ["preprod", "prod", "prodmr", "ovs", "ovsprod", "pr", "europe", "uat", "qa", "dev", "sandbox"]:
        if re.search(rf"\b{re.escape(e)}\b", tl):
            env_names.append(e)
    env_names = dedup_keep_order(env_names)

    selected_env = None
    selected_root = None
    if env_urls:
        selected_root = env_urls[0]
        for k, v in cfg.ENV_URLS.items():
            if v.lower() == selected_root:
                selected_env = k
                break
        if not selected_env:
            if "preprod" in selected_root:
                selected_env = "preprod"
            elif "ovsprod" in selected_root:
                selected_env = "ovs"
            elif "files-io-pr." in selected_root:
                selected_env = "pr"
            elif ".eu1-" in selected_root:
                selected_env = "europe"
            elif "prodmr" in selected_root:
                selected_env = "prodmr"
            elif "files-io-prod." in selected_root:
                selected_env = "prod"
    elif env_names:
        e = env_names[0]
        alias = {"ovsprod": "ovs", "production": "prod", "stage": "preprod"}
        selected_env = alias.get(e, e)
        selected_root = cfg.ENV_URLS.get(selected_env, cfg.DEFAULT_ROOT)

    if not selected_env:
        selected_env = cfg.DEFAULT_ENV
    if not selected_root:
        selected_root = cfg.ENV_URLS.get(selected_env, cfg.DEFAULT_ROOT)

    tenant_candidates = []
    tenant_candidates += re.findall(r"\btenant(?:\s*id)?\s*[:=]\s*([a-z0-9_-]{3,64})\b", tl)
    tenant_candidates += re.findall(r"processserver-\d+/([a-z0-9_-]{3,64})/", tl)
    tenant_candidates += re.findall(r"/([a-z0-9_-]{3,64})/[a-z0-9._-]{2,128}/", tl)
    tenant_ids = dedup_keep_order(tenant_candidates)

    store_candidates = []
    store_candidates += re.findall(r"\bstore\s*[:=]\s*([a-zA-Z0-9._-]{2,128})\b", t, flags=re.IGNORECASE)
    store_candidates += re.findall(r"\b(?:site)\s*[:=]?\s*\(?(R\d{3,8})\)?\b", t, flags=re.IGNORECASE)
    store_candidates += re.findall(r"\b(R\d{3,8})\b", t, flags=re.IGNORECASE)
    store_candidates += re.findall(r"/[a-z0-9_-]{3,64}/([a-zA-Z0-9._-]{2,128})/", t)
    store_ids = dedup_keep_order(store_candidates)

    process_servers = dedup_keep_order(re.findall(r"\bprocessserver-\d+\b", tl))
    if not process_servers:
        process_servers = cfg.PROCESS_SERVERS[:]

    # device IDs and UUIDs
    device_ids = []
    device_ids += re.findall(r"\bdevice\s*id\s*[:=]\s*([A-Za-z0-9-]{8,100})\b", t, flags=re.IGNORECASE)
    uuid_ids = re.findall(r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b", t)
    device_ids += uuid_ids
    device_ids += re.findall(r"\b([A-Z0-9]{8,}-[A-Z0-9]{4,}-[A-Z0-9]{4,}-[A-Z0-9]{4,}-[A-Z0-9]{8,})\b", t, flags=re.IGNORECASE)
    device_ids = dedup_keep_order([normalize_hyphen_spaces(d).upper() for d in device_ids])

    # dates
    ymd = re.findall(r"\b(20\d{2}-\d{2}-\d{2})\b", t)
    dates = dedup_keep_order(ymd)

    # sub paths from explicit path-like strings
    sub_paths = []
    sub_paths += re.findall(r"(/[^ \n]+/)", t)
    # derived tenant/store
    if tenant_ids and store_ids:
        sub_paths.append(f"/{tenant_ids[0]}/{store_ids[0]}/")
    if tenant_ids:
        sub_paths.append(f"/{tenant_ids[0]}/")
    # include configured defaults
    sub_paths += cfg.DEFAULT_SUB_PATHS
    sub_paths = dedup_keep_order(sub_paths)

    return {
        "env": selected_env,
        "root_url": selected_root,
        "tenant_ids": tenant_ids,
        "store_ids": store_ids,
        "process_servers": process_servers,
        "device_ids": device_ids,
        "uuid_ids": dedup_keep_order(uuid_ids),
        "dates": dates,
        "sub_paths": sub_paths
    }

# ---------------------------------------------------------------------------
# Log retrieval (Pattern C = A + B)
# ---------------------------------------------------------------------------
class LogFetcher:
    _FNAME_RE = re.compile(r"(\d{4}-\d{2}-\d{2})-(\d{2})-(\d{2})-\d+\.log\.gz$", re.IGNORECASE)

    def __init__(self, cfg: Config):
        self.cfg = cfg
        os.makedirs(cfg.DOWNLOAD_DIR, exist_ok=True)

    def _is_match_by_date(self, href: str, target_dates: List[str]) -> bool:
        if not href.endswith(".gz"):
            return False
        m = self._FNAME_RE.search(href)
        if m:
            d = m.group(1)
            return d in target_dates
        return any(d in href for d in target_dates)

    def _download_gz_content(self, url: str) -> str:
        try:
            r = no_cache_get(url, timeout=30)
            r.raise_for_status()
            raw = r.content
            if len(raw) < self.cfg.MIN_VALID_SIZE:
                return ""
            return gzip.decompress(raw).decode("utf-8", errors="replace")
        except Exception:
            return ""

    def _scrape_hrefs(self, url: str) -> List[str]:
        try:
            resp = no_cache_get(url, timeout=15)
            if resp.status_code != 200:
                return []
            if not HAS_BS4:
                return []
            soup = BeautifulSoup(resp.content, "html.parser")
            return [a["href"] for a in soup.find_all("a", href=True)]
        except Exception:
            return []

    def _try_pattern_a(self, root: str, server: str, tenant: str, store: str, device: str, dates: List[str]) -> Tuple[str, int]:
        # /<server>/<tenant>/<store>/<device>/
        if not tenant or not store or not device:
            return "", 0
        base = urljoin(root, f"{server}/{tenant}/{store}/{device}/")
        hrefs = self._scrape_hrefs(base)
        if not hrefs:
            return "", 0

        merged, count = [], 0
        for href in hrefs:
            if self._is_match_by_date(href, dates):
                full = urljoin(base, href)
                content = self._download_gz_content(full)
                if content:
                    count += 1
                    for ln in content.splitlines():
                        merged.append(f"[A|{server}|{tenant}|{store}|{device}|{href}] {ln}")
        return "\n".join(merged), count

    def _try_pattern_b(self, root: str, server: str, sub_path: str, ident: str, dates: List[str]) -> Tuple[str, int]:
        # /<server>/<sub_path>/<uuid_or_device>/
        sp = sub_path
        if not sp.startswith("/"):
            sp = "/" + sp
        if not sp.endswith("/"):
            sp += "/"
        base = urljoin(root, f"{server}{sp}{ident}/")
        hrefs = self._scrape_hrefs(base)
        if not hrefs:
            return "", 0

        merged, count = [], 0
        for href in hrefs:
            if self._is_match_by_date(href, dates):
                full = urljoin(base, href)
                content = self._download_gz_content(full)
                if content:
                    count += 1
                    for ln in content.splitlines():
                        merged.append(f"[B|{server}|{sp}|{ident}|{href}] {ln}")
        return "\n".join(merged), count

    def fetch_best_logs(self, entities: Dict[str, Any]) -> Dict[str, Any]:
        root = entities["root_url"]
        servers = entities["process_servers"] or self.cfg.PROCESS_SERVERS
        dates = entities["dates"] or default_date_candidates(self.cfg.DEFAULT_LOOKBACK_DAYS)

        tenant = entities["tenant_ids"][0] if entities["tenant_ids"] else ""
        store = entities["store_ids"][0] if entities["store_ids"] else ""
        devices = entities["device_ids"][:]
        idents_b = dedup_keep_order(entities["uuid_ids"] + entities["device_ids"] + entities["tenant_ids"])
        sub_paths = entities["sub_paths"][:]

        best = {"content": "", "files": 0, "route": None}

        # Pattern A attempts
        for server in servers:
            for dev in devices:
                content, cnt = self._try_pattern_a(root, server, tenant, store, dev, dates)
                if cnt > best["files"]:
                    best = {"content": content, "files": cnt, "route": f"A:{server}/{tenant}/{store}/{dev}"}

        # Pattern B attempts
        for server in servers:
            for sp in sub_paths:
                for ident in idents_b:
                    content, cnt = self._try_pattern_b(root, server, sp, ident, dates)
                    if cnt > best["files"]:
                        best = {"content": content, "files": cnt, "route": f"B:{server}{sp}{ident}"}

        return {
            "success": best["files"] > 0,
            "root_url": root,
            "dates_used": dates,
            "files_downloaded": best["files"],
            "chosen_route": best["route"],
            "merged_log_content": best["content"]
        }

# ---------------------------------------------------------------------------
# Classifier / Emotion / Response
# ---------------------------------------------------------------------------
class TicketClassifier:
    def __init__(self):
        self.intent_classifier = None
        self.sentiment_analyzer = None
        self.categories = [
            "Authentication/Access", "Network/Connectivity", "Database",
            "Email", "VPN", "Hardware", "Software", "Performance",
            "Security", "Billing", "Documentation", "Feature Request"
        ]
        self.priority_rules = {
            "P1": {"keywords": ["down", "outage", "critical", "emergency", "total loss"], "min_users": 100},
            "P2": {"keywords": ["broken", "major", "urgent", "not working", "failure"], "min_users": 10},
            "P3": {"keywords": ["slow", "error", "issue", "problem", "degraded"], "min_users": 1},
            "P4": {"keywords": ["minor", "typo", "formatting", "cosmetic"], "min_users": 1},
            "P5": {"keywords": ["feature request", "enhancement", "improvement"], "min_users": 0},
        }
        self.routing_map = {
            "Authentication/Access": "Identity Team",
            "Network/Connectivity": "Network Team",
            "Database": "Database Team",
            "Email": "Email Team",
            "VPN": "Network Team",
            "Hardware": "Hardware Team",
            "Software": "Application Team",
            "Performance": "Performance Team",
            "Security": "Security Team",
            "Billing": "Finance Team",
            "Documentation": "Documentation Team",
            "Feature Request": "Product Team",
        }

        if HAS_TRANSFORMERS:
            try:
                self.intent_classifier = pipeline(
                    "zero-shot-classification",
                    model="cross-encoder/nli-distilroberta-base",
                    device=NLP_DEVICE
                )
                self.sentiment_analyzer = pipeline(
                    "sentiment-analysis",
                    model="distilbert-base-uncased-finetuned-sst-2-english",
                    device=NLP_DEVICE
                )
            except Exception as e:
                logger.warning(f"Transformers init failed; fallback mode: {e}")

    def classify(self, summary: str, description: str, affected_users: int = 1) -> Dict[str, Any]:
        text = f"{summary} {description}"
        category, conf = self._determine_category(text)
        sentiment, sentiment_score = self._analyze_sentiment(text)
        priority = self._determine_priority(text, sentiment, affected_users)
        return {
            "category": category,
            "category_confidence": round(conf, 3),
            "priority": priority,
            "sentiment": sentiment,
            "sentiment_score": round(sentiment_score, 3),
            "affected_users": affected_users,
            "team": self.routing_map.get(category, "General Support"),
            "tags": [category.replace(" ", "_").replace("/", "_"), priority, "automated_triage"]
        }

    def _determine_category(self, text: str) -> Tuple[str, float]:
        if self.intent_classifier:
            try:
                r = self.intent_classifier(
                    text,
                    candidate_labels=self.categories,
                    hypothesis_template="This ticket is about {}."
                )
                return r["labels"][0], float(r["scores"][0])
            except Exception:
                pass
        tl = text.lower()
        for c in self.categories:
            kws = c.lower().replace("/", " ").split()
            if any(k in tl for k in kws):
                return c, 0.6
        return "Software", 0.5

    def _analyze_sentiment(self, text: str) -> Tuple[str, float]:
        if self.sentiment_analyzer:
            try:
                r = self.sentiment_analyzer(text[:512])[0]
                return r["label"].upper(), float(r["score"])
            except Exception:
                pass
        tl = text.lower()
        if any(w in tl for w in ["angry", "frustrated", "broken", "terrible", "awful", "furious", "down", "outage"]):
            return "NEGATIVE", 0.8
        if any(w in tl for w in ["great", "thanks", "happy", "satisfied", "perfect"]):
            return "POSITIVE", 0.8
        return "NEUTRAL", 0.5

    def _determine_priority(self, text: str, sentiment: str, affected_users: int) -> str:
        tl = text.lower()
        for p in ["P1", "P2", "P3", "P4", "P5"]:
            rule = self.priority_rules[p]
            if any(k in tl for k in rule["keywords"]) and affected_users >= rule["min_users"]:
                return p
        return "P2" if sentiment == "NEGATIVE" else "P4"

class EmotionDetector:
    def __init__(self):
        self.sentiment_model = None
        if HAS_TRANSFORMERS:
            try:
                self.sentiment_model = pipeline(
                    "sentiment-analysis",
                    model="distilbert-base-uncased-finetuned-sst-2-english",
                    device=NLP_DEVICE
                )
            except Exception:
                pass

    def analyze(self, summary: str, description: str) -> Dict[str, Any]:
        text = f"{summary} {description}"
        sentiment, score, emo, emo_scores = self._infer(text)
        esc = self._escalation(sentiment, emo)
        reason = "No escalation needed"
        if emo == "angry":
            reason = "Customer is angry - immediate attention required"
        elif emo == "frustrated":
            reason = "Customer is frustrated - escalate to senior support"
        elif sentiment == "NEGATIVE":
            reason = "Negative sentiment detected - requires human intervention"
        return {
            "sentiment": sentiment,
            "sentiment_score": round(score, 2),
            "dominant_emotion": emo,
            "emotion_scores": emo_scores,
            "escalation_score": round(esc, 2),
            "should_escalate": esc >= 0.5,
            "escalation_reason": reason
        }

    def _infer(self, text: str):
        if self.sentiment_model:
            try:
                r = self.sentiment_model(text[:512])[0]
                label = r["label"].upper()
                score = float(r["score"])
                tl = text.lower()
                if label == "NEGATIVE":
                    if any(w in tl for w in ["angry", "furious", "outraged"]):
                        emo = "angry"
                    elif any(w in tl for w in ["frustrated", "irritated", "annoyed"]):
                        emo = "frustrated"
                    else:
                        emo = "frustrated"
                    return "NEGATIVE", score, emo, {emo: round(score, 2), "neutral": round(1-score, 2)}
                elif label == "POSITIVE":
                    return "POSITIVE", score, "satisfied", {"satisfied": round(score, 2), "neutral": round(1-score, 2)}
                return "NEUTRAL", score, "neutral", {"neutral": round(score, 2)}
            except Exception:
                pass
        tl = text.lower()
        if any(w in tl for w in ["angry", "furious", "terrible", "awful"]):
            return "NEGATIVE", 0.85, "angry", {"angry": 0.85, "neutral": 0.15}
        if any(w in tl for w in ["frustrated", "broken", "not working", "down"]):
            return "NEGATIVE", 0.75, "frustrated", {"frustrated": 0.75, "neutral": 0.25}
        if any(w in tl for w in ["thanks", "great", "happy", "satisfied"]):
            return "POSITIVE", 0.8, "satisfied", {"satisfied": 0.8, "neutral": 0.2}
        return "NEUTRAL", 0.5, "neutral", {"neutral": 0.5}

    def _escalation(self, sentiment: str, emotion: str) -> float:
        score = 0.4 if sentiment == "NEGATIVE" else 0.0
        score += 0.5 * {"angry": 0.9, "frustrated": 0.7, "confused": 0.4, "satisfied": 0.0, "neutral": 0.1}.get(emotion, 0.3)
        return min(score, 1.0)

class ResponseGenerator:
    def __init__(self):
        self.templates = {
            "angry": """Dear {name},

Thank you for bringing this critical issue to our attention, and we sincerely apologize for the frustration.
Your ticket {ticket} has been marked as **{priority}** and immediately escalated to our senior team.

Best regards,
Support Team""",
            "frustrated": """Dear {name},

Thank you for reporting this issue. We appreciate your patience.
Your ticket {ticket} (Category: {category}, Priority: {priority}) has been prioritized.

Best regards,
Support Team""",
            "satisfied": """Dear {name},

Thank you for contacting us! We're glad to assist you.
Your ticket {ticket} is being processed and you'll hear from us shortly.

Best regards,
Support Team""",
            "default": """Dear {name},

Thank you for reaching out. Your ticket {ticket} has been received.
Assigned Team: {team}
You can expect an update within 2 hours.

Best regards,
Support Team"""
        }

    def generate(self, emotion: str, name: str, ticket: str, priority: str, category: str, team: str) -> str:
        tpl = self.templates.get(emotion, self.templates["default"])
        return tpl.format(name=name, ticket=ticket, priority=priority, category=category, team=team)

# ---------------------------------------------------------------------------
# Relevant log analyzer + RCA
# ---------------------------------------------------------------------------
class RelevantLogAnalyzer:
    def __init__(self):
        self.error_patterns = {
            "database": ["connection refused", "connection timeout", "database error", "db error", "sql exception"],
            "auth": ["authentication failed", "unauthorized", "invalid credentials", "forbidden", "token expired"],
            "network": ["connection reset", "timeout", "unreachable", "dns", "socket"],
            "memory": ["out of memory", "memory leak", "oom", "heap space"],
            "server_error": ["internal server error", "500", "exception", "traceback", "fatal"]
        }

    def _keywords(self, context: str) -> List[str]:
        tokens = re.findall(r"[a-zA-Z0-9_\-]{3,}", (context or "").lower())
        stop = {"the","and","for","with","from","that","this","your","ticket","issue","problem","customer","store","site","device"}
        return dedup_keep_order([t for t in tokens if t not in stop and not t.isdigit()])[:80]

    @staticmethod
    def _level(line: str) -> str:
        u = line.upper()
        if any(k in u for k in ["ERROR", "CRITICAL", "FATAL", "EXCEPTION", "TRACEBACK"]):
            return "ERROR"
        if "WARN" in u:
            return "WARN"
        return "INFO"

    def _score(self, line: str, keywords: List[str], entities: Dict[str, Any]) -> float:
        l = line.lower()
        s = 0.0

        s += min(sum(1 for k in keywords if k in l) * 0.2, 1.8)

        if any(t in l for t in ["error", "critical", "fatal", "exception", "traceback"]):
            s += 0.9
        elif "warn" in l:
            s += 0.35

        for d in entities.get("device_ids", [])[:5]:
            if d.lower() in l:
                s += 1.0
        for ten in entities.get("tenant_ids", [])[:5]:
            if ten.lower() in l:
                s += 0.9
        for st in entities.get("store_ids", [])[:5]:
            if st.lower() in l:
                s += 0.7
        for ip in entities.get("ips", [])[:5]:
            if ip.lower() in l:
                s += 0.7

        for pats in self.error_patterns.values():
            if any(p in l for p in pats):
                s += 0.7
                break
        return s

    def filter_relevant(self, merged_log_content: str, context_text: str, entities: Dict[str, Any], min_score: float = 0.9) -> List[Dict[str, Any]]:
        if not merged_log_content:
            return []
        kws = self._keywords(context_text)
        out = []
        for line in merged_log_content.splitlines():
            line = line.strip()
            if not line:
                continue
            sc = self._score(line, kws, entities)
            if sc >= min_score:
                out.append({
                    "timestamp": datetime.now().isoformat(),
                    "message": line,
                    "level": self._level(line),
                    "relevance_score": round(sc, 3)
                })
        out.sort(key=lambda x: x["relevance_score"], reverse=True)
        return out

    def summarize_errors(self, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
        errors = [r for r in rows if r["level"] == "ERROR"]
        cats: Dict[str, int] = {}
        for e in errors:
            msg = e["message"].lower()
            matched = False
            for c, pats in self.error_patterns.items():
                if any(p in msg for p in pats):
                    cats[c] = cats.get(c, 0) + 1
                    matched = True
                    break
            if not matched:
                cats["unknown"] = cats.get("unknown", 0) + 1
        return {
            "total_relevant_entries": len(rows),
            "total_errors": len(errors),
            "error_rate": (len(errors) / len(rows)) if rows else 0.0,
            "errors_by_category": cats,
            "top_errors": [e["message"] for e in errors[:10]]
        }

    def generate_rca(self, issue_key: str, summary: str, error_summary: Dict[str, Any]) -> Dict[str, Any]:
        cats = error_summary.get("errors_by_category", {})
        top = max(cats, key=cats.get) if cats else "unknown"
        root = {
            "database": "Database connectivity/query instability",
            "auth": "Authentication/authorization failure",
            "network": "Network timeout/connectivity issue",
            "memory": "Memory pressure / OOM",
            "server_error": "Unhandled application/server exception",
            "unknown": "No dominant signature found in relevant logs"
        }.get(top, "No dominant signature found in relevant logs")
        te = error_summary.get("total_errors", 0)
        severity = "HIGH" if te >= 10 else "MEDIUM" if te >= 1 else "LOW"
        return {
            "ticket_id": issue_key,
            "issue_summary": summary,
            "analysis_time": datetime.now().isoformat(),
            "severity": severity,
            "root_cause": root,
            "evidence": [
                f"Relevant log entries: {error_summary.get('total_relevant_entries', 0)}",
                f"Relevant errors: {te}",
                f"Error rate in relevant logs: {error_summary.get('error_rate', 0):.2%}",
                f"Dominant category: {top}"
            ],
            "recommendations": [
                "Validate affected component/service health",
                "Correlate with recent deployment/config changes",
                "Add proactive alerts for recurring signature",
                "Document fix in knowledge base"
            ]
        }

# ---------------------------------------------------------------------------
# KB + Incident prediction
# ---------------------------------------------------------------------------
class KBArticleGenerator:
    def __init__(self):
        self.resolution_keywords = ["fixed", "resolved", "solution", "workaround", "steps", "restart", "patched", "updated", "reconfigured"]

    def extract_resolution(self, ticket_data: Dict[str, Any], rca: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        desc = ticket_data.get("description", "") or ""
        lines = [ln.strip("-• ").strip() for ln in desc.splitlines() if ln.strip()]
        steps = [ln for ln in lines if any(k in ln.lower() for k in self.resolution_keywords)]
        if not steps and rca and rca.get("recommendations"):
            steps = rca["recommendations"]
        if not steps:
            steps = ["Investigated issue and applied standard remediation workflow."]
        return {
            "ticket_id": ticket_data.get("key"),
            "title": ticket_data.get("summary"),
            "category": ticket_data.get("category", "General"),
            "priority": ticket_data.get("priority", "P3"),
            "steps": steps,
            "status": "resolved"
        }

    def generate_kb_article(self, resolution: Dict[str, Any], rca: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        content = []
        content.append({"heading": "Problem", "items": [resolution.get("title", "N/A")]})
        content.append({"heading": "Solution", "items": resolution.get("steps", [])})
        if rca:
            content.append({"heading": "RCA Evidence", "items": [f"Root Cause: {rca.get('root_cause', 'N/A')}"] + rca.get("evidence", [])})
            content.append({"heading": "Prevention", "items": rca.get("recommendations", [])})
        return {
            "id": f"KB-{int(datetime.now().timestamp())}",
            "title": f"[{resolution.get('category')}] {resolution.get('title')}",
            "summary": f"Solution for {resolution.get('category')} issues",
            "category": resolution.get("category"),
            "priority": resolution.get("priority"),
            "content": content,
            "metadata": {
                "created_from_tickets": [resolution.get("ticket_id")],
                "created_at": datetime.now().isoformat(),
                "status": "draft"
            }
        }

class IncidentPredictor:
    def __init__(self, anomaly_threshold: float = 0.7):
        self.anomaly_threshold = anomaly_threshold

    def analyze_metrics(self, metrics: Dict[str, List[float]]) -> Dict[str, Any]:
        analysis = {
            "timestamp": datetime.now().isoformat(),
            "metrics_analyzed": len(metrics),
            "anomalies": [],
            "risk_score": 0.0,
            "incident_probability": 0.0
        }
        scores = []
        for name, values in metrics.items():
            if len(values) < 3:
                continue
            arr = np.array(values, dtype=float)
            mean = float(np.mean(arr))
            std = float(np.std(arr))
            cur = float(arr[-1])
            z = abs((cur - mean) / std) if std > 0 else 0.0
            a = min(z / 3, 1.0)
            if a > self.anomaly_threshold:
                analysis["anomalies"].append({
                    "metric": name,
                    "current_value": cur,
                    "mean": mean,
                    "z_score": round(z, 3),
                    "anomaly_score": round(a, 3),
                    "severity": "CRITICAL" if z > 3 else "HIGH"
                })
                scores.append(a)
        if scores:
            analysis["risk_score"] = float(np.mean(scores))
            analysis["incident_probability"] = float(min(analysis["risk_score"] * 1.5, 1.0))
        return analysis

    def predict_incidents(self, metrics: Dict[str, List[float]], hours_ahead: int = 4) -> List[Dict[str, Any]]:
        preds = []
        thresholds = {"cpu_usage": 85.0, "memory_usage": 90.0, "error_rate": 5.0}
        for name, values in metrics.items():
            if len(values) < 2 or name not in thresholds:
                continue
            arr = np.array(values, dtype=float)
            trend = float(np.polyfit(np.arange(len(arr)), arr, 1)[0])
            if trend > 0:
                future = float(arr[-1] + trend * hours_ahead)
                if future > thresholds[name]:
                    preds.append({
                        "metric": name,
                        "current_value": float(arr[-1]),
                        "predicted_value": future,
                        "threshold": thresholds[name],
                        "hours_until_breach": hours_ahead
                    })
        return preds

    def generate_alert(self, analysis: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        if not analysis.get("anomalies"):
            return None
        return {
            "alert_id": f"PRED-{int(datetime.now().timestamp())}",
            "severity": "HIGH" if analysis["incident_probability"] > 0.7 else "MEDIUM",
            "timestamp": analysis["timestamp"],
            "title": f"Potential incident detected - Risk {analysis['risk_score']:.2f}",
            "description": f"Detected {len(analysis['anomalies'])} anomaly/anomalies",
            "probability": analysis["incident_probability"]
        }

# ---------------------------------------------------------------------------
# Orchestrator pipeline
# ---------------------------------------------------------------------------
class AutomationPipeline:
    def __init__(self, cfg: Config):
        logger.info("Initializing pipeline...")
        self.cfg = cfg
        self.jira = JiraIntegration(cfg.JIRA_URL, cfg.JIRA_EMAIL, cfg.JIRA_API_TOKEN, cfg.JIRA_PROJECT_KEY)
        self.ocr = ScreenshotOCR(cfg.ATTACHMENT_DIR)
        self.fetcher = LogFetcher(cfg)

        self.classifier = TicketClassifier()
        self.emotion = EmotionDetector()
        self.response = ResponseGenerator()
        self.relevant_logs = RelevantLogAnalyzer()
        self.kb = KBArticleGenerator()
        self.predictor = IncidentPredictor()
        logger.info("Pipeline initialized.")

    @staticmethod
    def infer_affected_users(context: str) -> int:
        tl = (context or "").lower()
        if any(k in tl for k in ["all customers", "all users", "global outage", "production down"]):
            return 500
        if any(k in tl for k in ["multiple users", "many users", "store impacted"]):
            return 25
        if any(k in tl for k in ["single user", "one user"]):
            return 1
        return 5

    def process_ticket_number(self, issue_key: str) -> Dict[str, Any]:
        if not self.jira.is_ready():
            raise RuntimeError("Jira not connected/configured.")

        start = time.time()
        result: Dict[str, Any] = {"issue_key": issue_key, "timestamp": datetime.now().isoformat(), "stages": {}}

        # 1) Jira ticket
        ticket = self.jira.fetch_ticket_context(issue_key)
        summary = ticket["summary"]
        description = ticket["description"]
        reporter = ticket["reporter"]
        comments = ticket["comments"]
        issue_obj = ticket["issue"]

        # 2) OCR attachments (if easyocr available)
        image_paths = self.ocr.download_images(self.jira.jira, issue_obj)
        ocr_text = self.ocr.ocr_images(image_paths)

        full_context = f"{summary}\n{description}\n" + "\n".join(comments) + "\n" + ocr_text

        # 3) Parse routing entities
        entities = parse_routing_entities(full_context, self.cfg)
        if not entities["dates"]:
            entities["dates"] = default_date_candidates(self.cfg.DEFAULT_LOOKBACK_DAYS)

        # additional extracted fields for relevance
        entities["ips"] = dedup_keep_order(re.findall(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", full_context))

        print("\n[DEBUG] ROUTING EXTRACTION")
        print(" env:", entities["env"])
        print(" root:", entities["root_url"])
        print(" tenant_ids:", entities["tenant_ids"])
        print(" store_ids:", entities["store_ids"])
        print(" process_servers:", entities["process_servers"])
        print(" device_ids:", entities["device_ids"][:5])
        print(" uuid_ids:", entities["uuid_ids"][:5])
        print(" sub_paths:", entities["sub_paths"][:5])
        print(" dates:", entities["dates"])

        # 4) Classification + Emotion + Response
        affected_users = self.infer_affected_users(full_context)
        cls = self.classifier.classify(summary, description + "\n" + ocr_text, affected_users)
        emo = self.emotion.analyze(summary, description + "\n" + ocr_text)
        ack = self.response.generate(
            emo["dominant_emotion"], reporter, issue_key,
            cls["priority"], cls["category"], cls["team"]
        )

        result["stages"]["classification"] = cls
        result["stages"]["emotion"] = emo
        result["stages"]["acknowledgment"] = ack

        # 5) Fetch logs using Pattern C (A + B)
        fetch = self.fetcher.fetch_best_logs(entities)
        merged_log_content = fetch["merged_log_content"]

        result["stages"]["log_fetch"] = {
            "success": fetch["success"],
            "root_url": fetch["root_url"],
            "dates_used": fetch["dates_used"],
            "files_downloaded": fetch["files_downloaded"],
            "chosen_route": fetch["chosen_route"]
        }

        # 6) Relevant log analysis + RCA
        relevant_rows, error_summary, rca = [], {}, None
        if merged_log_content:
            relevant_rows = self.relevant_logs.filter_relevant(
                merged_log_content, full_context, entities, min_score=0.9
            )
            error_summary = self.relevant_logs.summarize_errors(relevant_rows)
            rca = self.relevant_logs.generate_rca(issue_key, summary, error_summary)

        result["stages"]["relevant_log_entries"] = relevant_rows[:150]
        result["stages"]["log_analysis"] = error_summary
        result["stages"]["rca"] = rca

        # 7) KB
        resolution = self.kb.extract_resolution({
            "key": issue_key,
            "summary": summary,
            "description": description,
            "category": cls["category"],
            "priority": cls["priority"]
        }, rca=rca)
        kb_article = self.kb.generate_kb_article(resolution, rca)
        result["stages"]["kb_article"] = kb_article

        # 8) Incident prediction (example metrics)
        metrics = {
            "cpu_usage": [45, 48, 50, 75, 82, 88, 92],
            "error_rate": [1.2, 1.3, 1.5, 3.0, 5.5, 7.2],
            "memory_usage": [60, 62, 65, 70, 85, 90]
        }
        analysis = self.predictor.analyze_metrics(metrics)
        predictions = self.predictor.predict_incidents(metrics)
        alert = self.predictor.generate_alert(analysis)
        result["stages"]["incident_prediction"] = {"analysis": analysis, "predictions": predictions, "alert": alert}

        # 9) Jira update
        jira_comment = (
            f"AI Triage Update\n"
            f"- Category: {cls['category']}\n"
            f"- Priority: {cls['priority']}\n"
            f"- Emotion: {emo['dominant_emotion']} (Escalate={emo['should_escalate']})\n"
            f"- Log Route: {fetch.get('chosen_route')}\n"
            f"- Logs Downloaded: {fetch.get('files_downloaded')}\n"
            f"- RCA: {(rca['root_cause'] if rca else 'Insufficient logs/data')}\n\n"
            f"Acknowledgment Draft:\n{ack}"
        )
        jira_ok = self.jira.update_ticket(issue_key, cls["priority"], cls["tags"], jira_comment)
        result["stages"]["jira_update"] = {"success": jira_ok}

        # 10) Efficiency
        runtime = round(time.time() - start, 2)
        result["stages"]["efficiency"] = {
            "automation_runtime_seconds": runtime,
            "mttr_improvement_signal": "HIGH" if cls["priority"] in ["P1", "P2"] and bool(rca) else "MEDIUM"
        }

        # cleanup attachments
        self.ocr.cleanup(image_paths)

        return result

    def generate_report(self, results: List[Dict[str, Any]]) -> Dict[str, Any]:
        total = len(results)
        successful = len([r for r in results if "error" not in r])
        p1 = len([r for r in results if r.get("stages", {}).get("classification", {}).get("priority") == "P1"])
        escalated = len([r for r in results if r.get("stages", {}).get("emotion", {}).get("should_escalate")])
        kb_generated = len([r for r in results if r.get("stages", {}).get("kb_article")])
        return {
            "total_tickets": total,
            "successful": successful,
            "p1_tickets": p1,
            "escalated": escalated,
            "kb_generated": kb_generated,
            "success_rate": (successful / total * 100) if total else 0.0
        }

# ---------------------------------------------------------------------------
# Main (ticket-number-only)
# ---------------------------------------------------------------------------
def print_compact_summary(result: Dict[str, Any]):
    s = result.get("stages", {})
    cls = s.get("classification", {})
    emo = s.get("emotion", {})
    fetch = s.get("log_fetch", {})
    rca = s.get("rca", {})
    eff = s.get("efficiency", {})
    print("\n" + "="*90)
    print(f"TICKET: {result.get('issue_key')}")
    print("="*90)
    print(f"Category/Priority   : {cls.get('category')} / {cls.get('priority')}")
    print(f"Emotion/Escalation  : {emo.get('dominant_emotion')} / {emo.get('should_escalate')}")
    print(f"Log fetch           : {fetch.get('success')} | route={fetch.get('chosen_route')} | files={fetch.get('files_downloaded')}")
    print(f"RCA                 : {(rca.get('root_cause') if rca else 'N/A')}")
    print(f"MTTR Signal         : {eff.get('mttr_improvement_signal')}")
    print("="*90 + "\n")

def main():
    print("\n" + "="*90)
    print("JSM INTELLIGENT SUPPORT AUTOMATION PLATFORM (FINAL C/C)")
    print("="*90 + "\n")

    pipeline = AutomationPipeline(config)
    if not pipeline.jira.is_ready():
        print("❌ Jira is not connected. Please set JIRA_EMAIL and JIRA_API_TOKEN.")
        sys.exit(1)

    all_results: List[Dict[str, Any]] = []

    while True:
        issue_key = input("Enter ticket number (e.g., IOC-123): ").strip()
        if not issue_key:
            print("Ticket number is required.")
            continue

        try:
            res = pipeline.process_ticket_number(issue_key)
            all_results.append(res)
            print_compact_summary(res)

            save = input("Save full JSON result? (y/n): ").strip().lower()
            if save == "y":
                fn = f"result_{issue_key}_{int(time.time())}.json"
                with open(fn, "w", encoding="utf-8") as f:
                    json.dump(res, f, indent=2)
                print(f"Saved: {fn}")

        except Exception as e:
            print(f"❌ Failed processing {issue_key}: {e}")

        again = input("Process another ticket? (y/n): ").strip().lower()
        if again != "y":
            break

    report = pipeline.generate_report(all_results)
    print("\nFINAL REPORT")
    print(f"Total Tickets: {report['total_tickets']}")
    print(f"Successful: {report['successful']}")
    print(f"P1 Critical: {report['p1_tickets']}")
    print(f"Escalated: {report['escalated']}")
    print(f"KB Articles: {report['kb_generated']}")
    print(f"Success Rate: {report['success_rate']:.1f}%")

if __name__ == "__main__":
    main()


JSM INTELLIGENT SUPPORT AUTOMATION PLATFORM (FINAL C/C)



Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Enter ticket number (e.g., IOC-123): IOC-4431



[DEBUG] ROUTING EXTRACTION
 env: preprod
 root: https://files-io-preprod.eus1-n.itemoptix.com/
 tenant_ids: []
 store_ids: []
 process_servers: ['processserver-0', 'processserver-1', 'processserver-2', 'processserver-3', 'processserver-4']
 device_ids: ['2D332E3A-C0A0-44BB-A1DD-280E8CC2FD0A']
 uuid_ids: ['2d332e3a-c0a0-44bb-a1dd-280e8cc2fd0a']
 sub_paths: ['//drive.google.com/file/d/1406CzovvcKrNsusXQsIf3GCkB-nlpLA8/', '//drive.google.com/file/d/1If2G_ydvYKbftlB3ueXI0qXU4EN34_bt/', '/jpretailerspp/5.11_TACTICAL_STORE_TOKYO/']
 dates: ['2026-08-12', '2026-08-11']

TICKET: IOC-4431
Category/Priority   : Software / P3
Emotion/Escalation  : satisfied / False
Log fetch           : False | route=None | files=0
RCA                 : N/A
MTTR Signal         : MEDIUM

Save full JSON result? (y/n): y
Saved: result_IOC-4431_1786562897.json
Process another ticket? (y/n): n

FINAL REPORT
Total Tickets: 1
Successful: 1
P1 Critical: 0
Escalated: 0
KB Articles: 1
Success Rate: 100.0%
